## 1. System Preparation
### IMPORT LIBRARIES

In [ ]:
!pip install hapiclient

In [ ]:
import pandas as pd
import numpy as np
from hapiclient.hapi import hapi
from hapiclient import hapitime2datetime
import matplotlib.pyplot as plt
import plotly.express as px
import scipy

### DEFINED FUNCTIONS


In [ ]:
# download dataset via HAPI
# return dataframe with all original columns with column 'Dataset'

def get_data(dataset):
    data, meta = hapi(server, dataset, parameters, start, stop, **opts)

    df = pd.DataFrame()
    for name in data.dtype.names:
        tmp = data[name]
        if tmp.shape[1:] == ():
            df[name] = tmp
        else:
            df2=pd.DataFrame(tmp, columns=[f"{name}_{i}" for i in range(tmp.shape[1])])
            df = pd.concat([df, df2], axis=1)
    df['Time'] = hapitime2datetime(data['Time'])
    df['Dataset'] = dataset
    return df

### SETTINGS
- set date range as `start` and `stop`
- settings for data downloading from HAPI

In [ ]:
start = '2021-04-01T00:00:00'
stop = '2021-05-01T00:00:00'

parameters = ''
opts = {'logging': True, 'usecache': True}
server = 'https://cdaweb.gsfc.nasa.gov/hapi'

## 2. Download STEREO-A data
... describing solar wind and magnetic field

In [ ]:
#STEREO-A in-situ
sta2 = get_data('STA_L2_MAGPLASMA_1M') #minute
sta2.head()

In [ ]:
sta2.columns

In [ ]:
#sta2=sta2[['Time', 'BTOTAL', 'Np', 'Vp', 'Tp', 'Entropy', 'Beta', 'Total_Pressure', 'Magnetic_Pressure', 'Dynamic_Pressure', 'R', 'Cone_Angle', 'Clock_Angle', 'Dataset']].copy()


Let's look at IMF

In [ ]:
fig = px.scatter(
    sta2,
    x='Time',
    y='BTOTAL',
    title='BTOTAL',
    width=800,
    height=400
)
fig.show()

we can see "incorrect" values

we will remove them like this...

In [ ]:
sta2['BTOTAL'] = np.where(sta2['BTOTAL'] < -100000, np.nan, sta2['BTOTAL'])  #IMPACT/MAG Total Magnetic Field [nT]

### - remove incorrect values


In [ ]:
# main parameters
sta2['Vp'] = np.where(sta2['Vp'] < 200, np.nan, sta2['Vp'])  #PLASTIC Proton Bulk Speed (proton) [km/s]
sta2['Np'] = np.where(sta2['Np'] < 0, np.nan, sta2['Np'])  #PLASTIC Solar Wind Proton Number Density (proton) [1/cm^3]
sta2['Tp'] = np.where(sta2['Tp'] < 0, np.nan, sta2['Tp'])  #PLASTIC Proton Temperature (proton) [K]

# pressures
sta2['Total_Pressure'] = np.where(sta2['Total_Pressure'] < 0, np.nan, sta2['Total_Pressure'])  #[pPa]
sta2['Magnetic_Pressure'] = np.where(sta2['Magnetic_Pressure'] < 0, np.nan, sta2['Magnetic_Pressure'])  #[pPa]
sta2['Dynamic_Pressure'] = np.where(sta2['Dynamic_Pressure'] < 0, np.nan, sta2['Dynamic_Pressure'])  #[nPa]

# other
sta2['Entropy'] = np.where(sta2['Entropy'] < 0, np.nan, sta2['Entropy']) #[]
sta2['Beta']=np.where(sta2['Beta'] < 0, np.nan, sta2['Beta'])
sta2['R'] = np.where(sta2['R'] < 0, np.nan, sta2['R']) #Distance of STEREO from the Sun [AU]
sta2['Cone_Angle'] = np.where(sta2['Cone_Angle'] < 0, np.nan, sta2['Cone_Angle']) #[deg]
sta2['Clock_Angle'] = np.where(sta2['Clock_Angle'] < -360, np.nan, sta2['Clock_Angle']) #[deg]

The total perpendicular pressure $P_t$ is the sum of the magnetic pressure and plasma thermal pressure perpendicular to the magnetic field:

$P_t = \frac{B^2}{2\mu_0}+n_ikT_i+n_ekT_e$, where $i$ are ions (protons, $\alpha$ particles) and $e$ are electrons.

### - convert to SI units

In [ ]:
sta2['BTOTAL_SI'] = sta2['BTOTAL']*1e-9 #nT to T
sta2['Vp_SI'] = sta2['Vp']*1e3 #km/s to m/s
sta2['Np_SI'] = sta2['Np']*1e6 #cm^-3 to m^-3
sta2['Total_Pressure_SI'] = sta2['Total_Pressure']*1e-12 #pPa to Pa
sta2['Magnetic_Pressure_SI'] = sta2['Magnetic_Pressure']*1e-12 #pPa to Pa
sta2['Dynamic_Pressure_SI'] = sta2['Dynamic_Pressure']*1e-9 #nPa to Pa

let's calculate magnetic pressure and compare with magnetic pressure already provided

In [ ]:
sta2['CAL_Magnetic_Pressure'] = ((sta2['BTOTAL_SI']*sta2['BTOTAL_SI'])
                                 / (2 * scipy.constants.physical_constants['vacuum mag. permeability'][0]))

In [ ]:
fig = px.scatter(
    sta2,
    x='Time',
    y=['Magnetic_Pressure_SI', 'CAL_Magnetic_Pressure'],
    title='Magnetic Pressure',
    labels={'Time': 'Date', 'value': 'Magnetic Pressure [Pa]', 'variable': 'Source'},
    width=800,
    height=400
)
fig.for_each_trace(lambda t: t.update(name={
    'Magnetic_Pressure_SI': 'STA dataset Authors',
    'CAL_Magnetic_Pressure': 'Calculated'
}[t.name]))
fig.show()

great, we have magnetic pressure, let's calculate thermal pressure and total pressure

In [ ]:
sta2['CAL_Thermal_Pressure_PROTON'] = sta2['Np_SI'] * sta2['Tp'] * scipy.constants.k
sta2['CAL_Thermal_Pressure'] = sta2['Np_SI'] * sta2['Tp'] * scipy.constants.k * (1.2+1.54e5/sta2['Tp'])

sta2['CAL_Total_Pressure_PROTON'] = sta2['CAL_Magnetic_Pressure'] + sta2['CAL_Thermal_Pressure_PROTON']
sta2['CAL_Total_Pressure'] = sta2['CAL_Magnetic_Pressure'] + sta2['CAL_Thermal_Pressure']

In [ ]:
fig = px.scatter(
    sta2[['Time', 'Total_Pressure_SI', 'CAL_Total_Pressure', 'CAL_Total_Pressure_PROTON']],
    x='Time',
    y=['Total_Pressure_SI', 'CAL_Total_Pressure', 'CAL_Total_Pressure_PROTON'],
    title='Total Pressure',
    labels={'Time': 'Date', 'value': 'Total Pressure [Pa]', 'variable': 'Source'},
    width=800,
    height=400
)
fig.for_each_trace(lambda t: t.update(name={
    'Total_Pressure_SI': 'STA dataset Authors',
    'CAL_Total_Pressure': 'Calculated (for all particles)',
    'CAL_Total_Pressure_PROTON': 'Calculated (for protons)'
}[t.name]))
fig.show()